# Guardrail 4 — Authorization

**Where it sits:** between retrieval result and prompt assembly. **This is the guard that makes multi-tenant RAG safe.**

**What it stops:** cross-tenant reads, role bypass, partial-tenant leakage.

**Decision contract:** `{allow | rewrite | block, visible_chunks[], redacted[], reasons[]}`

**Self-contained:** inlines a tiny multi-tenant RAG. No imports from other folders.

## Step 1 — toy multi-tenant RAG scaffold

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the same directory as this notebook (works in Jupyter)
_env_path = Path.cwd() / ".env"
if not _env_path.exists():
    _env_path = Path(__file__).parent / ".env" if "__file__" in globals() else _env_path
load_dotenv(_env_path, override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

# LangChain primitives, all driven by .env
LLM_MODEL     = os.getenv("MINIMAX_MODEL", "MiniMax-M3")
LLM_BASE_URL  = os.getenv("MINIMAX_BASE_URL", "https://api.minimax.io/v1")
LLM_API_KEY   = os.getenv("MINIMAX_API_KEY", "")

llm = ChatOpenAI(
    model=LLM_MODEL,
    api_key=LLM_API_KEY or "sk-fake",   # placeholder if no key -- calls will fail loudly
    base_url=LLM_BASE_URL,
    temperature=0,
)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=LLM_API_KEY or "sk-fake",
    base_url=LLM_BASE_URL,
)

print(f"LLM configured:  model={LLM_MODEL}  base_url={LLM_BASE_URL}")
print(f"API key loaded:  {'yes ('+LLM_API_KEY[:8]+'...)' if LLM_API_KEY else 'NO -- calls will fail; toy fallbacks below are unaffected'}")

# A safe wrapper so toy guardrail tests below stay deterministic.
# If FAKE_LLM=1 (or no key), use the toy. Otherwise call the real one.
_USE_FAKE = os.getenv("FAKE_LLM", "1") == "1" or not LLM_API_KEY

def chat(prompt: str, system: str = None) -> str:
    """Invoke the LangChain ChatOpenAI. Returns .content."""
    if _USE_FAKE:
        raise RuntimeError("chat() called but FAKE_LLM=1 -- use the toy LLM in this notebook's tests")
    msgs = []
    if system:
        msgs.append(SystemMessage(content=system))
    msgs.append(HumanMessage(content=prompt))
    return llm.invoke(msgs).content


In [ ]:
import re, math, hashlib

# Documents have a tenant and a set of ACLs (roles). "*" means public.
DOCS = [
    {"id": "d1", "text": "The capital of France is Paris.",
     "tenant": "public", "acls": {"*"}},
    {"id": "d2", "text": "Q3 revenue was $4.2M, up 12% YoY. CFO: J. Park.",
     "tenant": "acme", "acls": {"role:cfo", "role:exec"}},
    {"id": "d3", "text": "Q3 pipeline is $11M. Top deal: Globex $3M.",
     "tenant": "acme", "acls": {"role:exec", "role:sales-lead"}},
    {"id": "d4", "text": "Employee SSN list: 111-22-3333, 222-33-4444.",
     "tenant": "acme", "acls": {"role:cfo", "role:hr"}},
    {"id": "d5", "text": "Globex forecast: 50% discount if renewed.",
     "tenant": "initech", "acls": {"role:exec"}},
]

def embed(text, dim=32):
    words = re.findall(r"[a-z0-9]+", text.lower())
    v = [0.0] * dim
    for w in words:
        h = int(hashlib.md5(w.encode()).hexdigest(), 16)
        v[h % dim] += 1.0
    n = math.sqrt(sum(x*x for x in v)) or 1.0
    return [x/n for x in v]

def retrieve(query, k=5):
    qv = embed(query)
    scored = sorted(DOCS, key=lambda d: -sum(x*y for x,y in zip(qv, embed(d["text"]))))
    return [{"id": d["id"], "text": d["text"], "tenant": d["tenant"], "acls": d["acls"]}
            for d in scored[:k]]

## Step 2 — authorization guardrail

In [ ]:
def authz_guard(chunks, identity: dict):
    """Apply tenant + ACL filtering to retrieved chunks.

    identity: {user_id, tenant, roles: list[str], attrs: dict}
    """
    reasons, visible, redacted = [], [], []

    # (a) identity must be present and complete
    if not identity or not identity.get("user_id") or not identity.get("tenant"):
        return {"decision": "block",
                "reasons": ["missing_identity"],
                "visible_chunks": [], "redacted": []}

    user_tenant = identity["tenant"]
    user_roles  = set(identity.get("roles", []))

    for c in chunks:
        doc_tenant = c.get("tenant")
        doc_acls   = c.get("acls", set())

        # (b) tenant isolation — only the user's tenant OR public
        if doc_tenant != "public" and doc_tenant != user_tenant:
            redacted.append({"id": c["id"], "reason": "wrong_tenant",
                              "doc_tenant": doc_tenant, "user_tenant": user_tenant})
            continue

        # (c) ACL intersection — wildcard OR matching role
        if "*" not in doc_acls and not (doc_acls & user_roles):
            redacted.append({"id": c["id"], "reason": "no_matching_role",
                              "required": sorted(doc_acls), "have": sorted(user_roles)})
            continue

        # (d) ATTRIBUTE check (e.g. time-window) — toy: reject if chunk has
        #     'forecast' in text and identity doesn't have attr 'sees_forecasts'
        if "forecast" in c["text"].lower() and not identity.get("attrs", {}).get("sees_forecasts"):
            redacted.append({"id": c["id"], "reason": "attr_missing:sees_forecasts"})
            continue

        visible.append(c)

    decision = "allow" if visible else ("rewrite" if chunks else "allow")
    return {"decision": decision, "visible_chunks": visible, "redacted": redacted, "reasons": reasons}

## Step 3 — test cases

In [ ]:
alice_acme_cfo = {"user_id": "u-alice", "tenant": "acme", "roles": ["role:cfo", "role:exec"]}
bob_acme_sales = {"user_id": "u-bob",   "tenant": "acme", "roles": ["role:sales-lead"]}
carol_initech  = {"user_id": "u-carol", "tenant": "initech", "roles": ["role:exec"], "attrs": {"sees_forecasts": True}}
no_identity    = {}

chunks = retrieve("Q3 revenue forecast")  # raw retrieval — no authz applied
print(f"raw retriever returned {len(chunks)} chunks:\n  " +
      "\n  ".join(f"{c['id']} tenant={c['tenant']} acls={c['acls']}" for c in chunks))

for label, ident in [("alice (acme cfo+exec)", alice_acme_cfo),
                     ("bob   (acme sales-lead)", bob_acme_sales),
                     ("carol (initech exec, sees forecasts)", carol_initech),
                     ("anonymous", no_identity)]:
    r = authz_guard(chunks, ident)
    print(f"\n=== {label} ===")
    print(f"  decision:   {r['decision']}")
    print(f"  visible:    {[c['id'] for c in r['visible_chunks']]}")
    print(f"  redacted:   {[(d['id'], d['reason']) for d in r['redacted']]}")

## Step 4 — the cross-tenant attack

In [ ]:
# Carol from initech tries to ask about acme's Q3 numbers.
r = authz_guard(retrieve("Q3 revenue"), carol_initech)
print("Carol visible:", [c['id'] for c in r['visible_chunks']])
print("Carol redacted:", [(d['id'], d['reason']) for d in r['redacted']])

# The Globex chunk is initech's, and Carol is exec with sees_forecasts — she gets it.
print("\nGlobex forecast chunk is tenant=initech, sees_forecasts=True → visible to Carol ✓")

# Bob (acme sales) does NOT have sees_forecasts → blocked from forecast text.
r = authz_guard([{"id": "d5", "text": "Globex forecast: 50% discount.",
                   "tenant": "initech", "acls": {"role:exec"}}], bob_acme_sales)
print("\nBob attempting forecast from initech doc:")
print("  visible:", [c['id'] for c in r['visible_chunks']])
print("  redacted:", [(d['id'], d['reason']) for d in r['redacted']])

In [ ]:
### Real LangChain demo: authz guard as a RunnableLambda on retrieved docs

from langchain_core.runnables import RunnableLambda

def _authz_filter(identity: dict, retriever):
    def _run(query: str):
        docs = retriever.invoke(query)
        r = authz_guard([{"id": d.metadata["id"], "text": d.page_content,
                            "tenant": d.metadata["tenant"], "acls": d.metadata["acls"]}
                          for d in docs], identity)
        return [d for d in docs if d.metadata["id"] in {c["id"] for c in r["visible_chunks"]}]
    return _run

if not _USE_FAKE:
    from langchain_community.vectorstores import FAISS
    from langchain_core.documents import Document
    lc_vs = FAISS.from_documents([Document(page_content=d["text"], metadata=d) for d in DOCS], embeddings)
    chain = RunnableLambda(_authz_filter(alice_acme_cfo, lc_vs.as_retriever()))
    out = chain.invoke("Q3 revenue")
    print(f"alice sees {len(out)} docs after authz")
else:
    print("[FAKE_LLM=1 -- skipping real index.]")


## Takeaways

- **Authorization is the guard you cannot retrofit.** If your index has no tenant/ACL fields, every guardrail you build on top is a polite suggestion. Schema it in from day one.
- **Tenant check first, ACL second.** A wrong-tenant document is wrong regardless of role. Reject it before evaluating ACLs.
- **Attributes extend roles cleanly.** Time-window, region, classification-level — they all fit the same shape: doc says `requires:X`, user has `attrs[X]` or doesn't.
- **Don't rewrite → "you don't have access."** Just drop the chunk. Telling the user the doc exists tells them something they shouldn't know.
- **Missing identity = block, not allow.** A request without identity is the most common attack vector. Never default to allowing anonymous.

**Negative fixture checklist:** cross-tenant read, missing role, missing identity, attribute-mismatch. ✓